# XML Data Ingestion with LakeLogic 📄

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/files/xml/xml_example.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/files/xml/xml_example.ipynb)

## Business Scenario
Your organization receives employee data exports from legacy HR systems in XML format. These files contain employee records with critical information like emails, salaries, and employment status.

Common data quality issues:
- Invalid email formats (missing @ symbol)
- Negative or zero salaries (data entry errors)
- Inactive employees included in active reports

## Value Proposition
- **Native XML Support**: No manual parsing required - point LakeLogic directly at `.xml` files
- **Engine Agnostic**: Works with Polars, Pandas, DuckDB, or Spark
- **Automatic Validation**: Catch data quality issues before they reach your warehouse
- **Clear Error Tracking**: Every quarantined record includes specific failure reasons

## 1. Setup
First, we'll install LakeLogic (if needed) and import the necessary libraries.

In [ ]:
# Uncomment to install LakeLogic with XML support
# %pip install lakelogic[polars]

import os
from pathlib import Path
from lakelogic import DataProcessor

# Optional: Set your preferred engine
os.environ["LAKELOGIC_ENGINE"] = "polars"  # or pandas, duckdb, spark

print("✅ LakeLogic ready for XML ingestion!")

## 2. Preview the XML Data
Let's take a quick look at the raw XML file to see what we're working with.

In [ ]:
# Display first few lines of XML
xml_path = Path("data/employees.xml")
with open(xml_path, "r") as f:
    content = f.read()
    print(content[:800])  # First 800 characters
    print("\n...")

## 3. Run LakeLogic with XML Source
Now we'll use the Data Contract to ingest, validate, and transform the XML data.

In [ ]:
# Initialize processor with contract
processor = DataProcessor(contract="xml_contract.yaml")

# Run ingestion - LakeLogic automatically detects .xml extension
result = processor.run_source("data/employees.xml")

print(f"\n📊 Processing Summary:")
print(f"   Raw records: {len(result.raw)}")
print(f"   ✅ Valid: {len(result.good)}")
print(f"   ❌ Quarantined: {len(result.bad)}")
print(f"\n{result}")

## 4. Inspect Raw XML Data
Let's see what the data looks like after XML parsing (before validation).

In [ ]:
print("📄 RAW DATA (Parsed from XML):")
print(result.raw)

## 5. Inspect Validated Records (Silver Layer)
These records passed all validation rules and transformations.

In [ ]:
print("✅ VALIDATED DATA (Silver Layer):")
print(result.good)
print(f"\nColumns after transformation: {result.good.columns}")

## 6. Inspect Quarantined Records
These records failed validation - let's see why.

In [ ]:
print("🚨 QUARANTINED DATA (Failed Validation):")
if len(result.bad) > 0:
    print(result.bad)
    print("\n📋 Error Details:")
    # Show specific error reasons
    for idx, row in enumerate(result.bad.iter_rows(named=True) if hasattr(result.bad, 'iter_rows') else result.bad.to_dict('records')):
        print(f"\nRecord {idx + 1}: {row.get('name', 'N/A')}")
        errors = row.get('_lakelogic_errors', [])
        for error in errors:
            print(f"  ❌ {error}")
else:
    print("No records were quarantined - all data is valid! 🎉")

## 7. Data Quality Metrics
Let's calculate some key metrics about our data quality.

In [ ]:
total = len(result.raw)
valid = len(result.good)
invalid = len(result.bad)
pass_rate = (valid / total * 100) if total > 0 else 0

print("📈 Data Quality Metrics")
print("=" * 40)
print(f"Total Records:     {total:>5}")
print(f"Valid Records:     {valid:>5}")
print(f"Failed Records:    {invalid:>5}")
print(f"Pass Rate:         {pass_rate:>5.1f}%")
print("=" * 40)

# Alert if pass rate is below threshold
if pass_rate < 80:
    print("\n⚠️  WARNING: Pass rate below 80% - investigate data quality issues!")
else:
    print("\n✅ Data quality looks good!")

## 8. Try Different Engines
LakeLogic supports multiple engines. Let's try switching engines to see the flexibility.

In [ ]:
# Try with different engine (uncomment the engine you want to test)
# os.environ["LAKELOGIC_ENGINE"] = "pandas"
# os.environ["LAKELOGIC_ENGINE"] = "duckdb"
# os.environ["LAKELOGIC_ENGINE"] = "spark"

processor_alt = DataProcessor(contract="xml_contract.yaml")
print(f"Current engine: {processor_alt.engine_name}")

result_alt = processor_alt.run_source("data/employees.xml")
print(f"Results: {result_alt}")

## Summary

In this tutorial, we demonstrated:

1. ✅ **Native XML Ingestion**: No manual parsing - LakeLogic automatically detected and read the `.xml` file
2. ✅ **Automatic Validation**: Email formats, salary ranges, and employment status were validated
3. ✅ **Clear Error Tracking**: Each quarantined record includes specific reasons for failure
4. ✅ **Data Enrichment**: Added `seniority_level` based on hire date
5. ✅ **Engine Flexibility**: Same contract works with Polars, Pandas, DuckDB, or Spark

### Next Steps

- **Modify the contract**: Try adding your own validation rules to `xml_contract.yaml`
- **Change the data**: Edit `employees.xml` to test different validation scenarios
- **Materialize results**: Use LakeLogic's materialization feature to persist validated data
- **Production deployment**: Integrate this into your data pipeline for automated XML ingestion